# Thermal Localized Power Viewer

Use the slider below to step through the coupled digital twin results and inspect the thermal power slice at a chosen `z` index.

The notebook expects `coupled_digital_twin_results.npz` to live in the same directory.

In [ ]:
from pathlib import Path

import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from ipywidgets import IntSlider, FloatSlider, Checkbox, interact

HERE = Path.cwd()
PACKAGE_ROOT = HERE.parent.parent
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from digital_twin_files.rom.digital_twin import reshape_flux_vector

RESULTS_PATH = Path("coupled_digital_twin_results.npz")
if not RESULTS_PATH.exists():
    raise FileNotFoundError(f"Could not find {RESULTS_PATH}. Run digital_twin_driver.py first.")

results = np.load(RESULTS_PATH)
localized_power = results["localized_power"]
macro_time = results["time"]

print(f"Loaded {localized_power.shape[0]} macro frames from {RESULTS_PATH}")
print(f"Time range: {macro_time[0]:.3f} s to {macro_time[-1]:.3f} s")


def show_frame(frame_idx: int, z_index: int = 35, micro: float = 0.0, normalize: bool = True, log_scale: bool = True) -> None:
    # Get base frame and optionally the next frame for interpolation
    n_frames = localized_power.shape[0]
    idx = int(frame_idx)
    field1 = reshape_flux_vector(localized_power[idx])
    if (idx + 1) < n_frames and micro > 0.0:
        field2 = reshape_flux_vector(localized_power[idx + 1])
        field = (1.0 - float(micro)) * field1 + float(micro) * field2
    else:
        field = field1

    thermal_slice = field[z_index, :, :, 0]
    positive_slice = np.maximum(thermal_slice, 1e-12)
    
    # Compute amplitude (max value of this frame)
    amplitude = float(np.max(positive_slice))
    
    if normalize:
        # Normalize to shape function [0, 1], show amplitude separately
        normalized_slice = positive_slice / amplitude
        norm = Normalize(vmin=0, vmax=1)
        cbar_label = "Normalized shape (0-1)"
    else:
        # Raw values with log scale
        normalized_slice = positive_slice
        if log_scale:
            norm = LogNorm(vmin=float(np.min(positive_slice[positive_slice > 0])) if np.any(positive_slice > 0) else 1e-12, 
                          vmax=amplitude)
            cbar_label = "Thermal power (log)"
        else:
            norm = Normalize(vmin=0, vmax=amplitude)
            cbar_label = "Thermal power"

    plt.figure(figsize=(7, 6))
    plt.imshow(
        normalized_slice,
        origin="lower",
        cmap="inferno",
        norm=norm,
    )
    plt.colorbar(label=cbar_label)
    title = f"Thermal localized power | frame={frame_idx}, t={macro_time[idx]:.3f} s, z={z_index}"
    if micro and (idx + 1) < n_frames:
        title += f", micro={micro:.2f}"
    if normalize:
        title += f" | amplitude={amplitude:.2e}"
    plt.title(title)
    plt.xlabel("X index")
    plt.ylabel("Y index")
    plt.tight_layout()
    plt.show()

Loaded 5 macro frames from coupled_digital_twin_results.npz
Time range: 0.000 s to 2.000 s


In [2]:
# compute z-range from first reshaped frame
field0 = reshape_flux_vector(localized_power[0])
z_max = field0.shape[0] - 1

frame_slider = IntSlider(min=0, max=localized_power.shape[0] - 1, step=1, value=0, description="Frame")
z_slider = IntSlider(min=0, max=z_max, step=1, value=min(35, z_max), description="Z index")

micro_slider = FloatSlider(min=0.0, max=1.0, step=0.01, value=0.0, description="Micro")
normalize_checkbox = Checkbox(value=True, description="Normalize shape")
log_scale_checkbox = Checkbox(value=True, description="Log scale (raw mode)")

interact(show_frame, frame_idx=frame_slider, z_index=z_slider, micro=micro_slider, normalize=normalize_checkbox, log_scale=log_scale_checkbox)

interactive(children=(IntSlider(value=0, description='Frame', max=4), IntSlider(value=35, description='Z index…

<function __main__.show_frame(frame_idx: int, z_index: int = 35, micro: float = 0.0, normalize: bool = True, log_scale: bool = True) -> None>

# Controls and usage

- **Frame:** step through macro frames (integer frames from the dataset).
- **Micro:** interpolate between the selected frame and the next frame; `0.0` shows the exact frame, `1.0` shows the next frame. Interpolation is ignored on the final frame.
- **Z index:** axial slice index (computed from data shape).
- **Normalize shape:** when checked, each frame is normalized to [0, 1] to show the shape function (colorbar is always 0–1). The actual amplitude is displayed in the title. When unchecked, shows raw power values.
- **Log scale (raw mode):** only used when **Normalize shape** is off. When checked, uses logarithmic scaling for the raw power values; when unchecked, uses linear scaling.

Notes:
- The viewer uses a small positive floor (`1e-12`) to avoid log-of-zero issues.
- When normalized, you see how the shape changes while the amplitude factor is reported separately.